In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
import sys
import json
from numpy import linalg as LA
import tensorflow as tf
import getpass
ui = getpass.getuser()
if ui == 'laura':
    p = '/home/laura'
elif ui == 'lauradriscoll':
    p = '/Users/lauradriscoll/Documents'
elif ui == 'slibkind':
    p = '/Users/slibkind/Documents/'
##ADD YOUR PATH TO CODE REPOS HERE

net = 'binary_inputs'
PATH_YANGNET = os.path.join(p,'code/sophie-nets',net) 
sys.path.insert(0, PATH_YANGNET)
from task import generate_trials, rule_name, rule_index_map, rules_dict
from network import Model, get_perf, FixedPoint_Model
import tools

In [2]:
#WHERE IS THE NETWORK
rnn_type = 'LeakyRNN'
activation = 'softplus'
w_init = 'randgauss'
ruleset = 'basic'
n_rnn = str(256)
l2w = -7.0
l2h = -7.0
l1w = 0
l1h = 0
lr = -7.0
seed = str(0)
rule_trains = [rules_dict['basic'][1],rules_dict['basic'][3]]
rule_trains_str = '_'.join(rule_trains)
sigma_rec = 1/20
sigma_x = 2/20
w_rec_coeff = .9

net_name = 'lr'+"{:.1f}".format(-lr)+'l2_w'+"{:.1f}".format(-l2w)+'_h'+"{:.1f}".format(-l2h)
net_name2 = '_sig_rec'+str(sigma_rec)+'_sig_x'+str(sigma_x)+'_w_rec_coeff'+"{:.1f}".format(w_rec_coeff)+'_'+rule_trains_str

dir_specific_all = os.path.join(ruleset,rnn_type,activation,
    w_init,str(len(rule_trains))+'_tasks',str(n_rnn)+'_n_rnn',net_name+net_name2)

m = os.path.join(p,'data','sophie-nets',net,'data',dir_specific_all,str(seed))

In [3]:
#THIS IS SOME GENERAL CODE THAT WILL GET YOU LOTS OF VARIABLES YOU'LL WANT TO LOOK AT

rule = 'fdgo' # SET A PARTICULAR TASK OF INTEREST

model = Model(m)
with tf.Session() as sess:
    model.restore()
    var_list = model.var_list
    params = [sess.run(var) for var in var_list]
    hparams = model.hp
    trial = generate_trials(rule, hparams, mode='random', noise_on=False,batch_size = 200)
    feed_dict = tools.gen_feed_dict(model, trial, hparams)
    h, y_hat = sess.run([model.h, model.y_hat], feed_dict=feed_dict) #HIDDEN STATE, OUTPUT (n_TIME, n_TRIAL, n_UNIT)

Variables being optimized:
<tf.Variable 'rnn/leaky_rnn_cell/kernel:0' shape=(263, 256) dtype=float32_ref>
<tf.Variable 'rnn/leaky_rnn_cell/bias:0' shape=(256,) dtype=float32_ref>
<tf.Variable 'output/weights:0' shape=(256, 3) dtype=float32_ref>
<tf.Variable 'output/biases:0' shape=(3,) dtype=float32_ref>
Instructions for updating:
Use `tf.global_variables_initializer` instead.
INFO:tensorflow:Restoring parameters from /Users/slibkind/Documents/data/sophie-nets/binary_inputs/data/basic/LeakyRNN/softplus/randgauss/2_tasks/256_n_rnn/lr7.0l2_w7.0_h7.0_sig_rec0.05_sig_x0.1_w_rec_coeff0.9_delaygo_delayanti/0/model.ckpt
Model restored from file: /Users/slibkind/Documents/data/sophie-nets/binary_inputs/data/basic/LeakyRNN/softplus/randgauss/2_tasks/256_n_rnn/lr7.0l2_w7.0_h7.0_sig_rec0.05_sig_x0.1_w_rec_coeff0.9_delaygo_delayanti/0/model.ckpt


In [24]:
n_input = hparams['n_input']
n_rnn = hparams['n_rnn']
n_output = hparams['n_output']
w_in = params[0]  # [input weights , recurrent weights] 7+256, 256
b_in = params[1]
w_out = params[2]
b_out = params[3]
sigma_rec = hparams['sigma_rec']
dt = hparams['dt']
tau = hparams['tau']
alpha = dt/tau
activation = hparams['activation']

if activation == 'softplus':
    _activation = lambda x: np.log(np.exp(x) + 1)
elif activation == 'tanh':
    _activation = lambda x: np.tanh(x)
elif activation == 'relu':
    _activation = lambda x: x * (x > 0)
elif activation == 'power':
    _activation = lambda x: (x * (x > 0))**2
elif activation == 'retanh':
    _activation = lambda x: np.tanh(x * (x > 0))

In [28]:
def rnn_vanilla(params, h, x, alpha):
    alpha = .2
    xh = np.concatenate([x,h], axis=0)
    gate_inputs = np.dot(params[0].T,xh)+params[1]
    noise = 0
    output = _activation(gate_inputs) # + noise

    h_new = (1-alpha) * h + alpha * output
    
    return h_new
    
def vanilla_run_with_h0(params, x_t, h0, alpha):
    h = h0
    h_t = []
    h_t.append(np.expand_dims(h0,axis=1))
    for x in x_t:
        h = rnn_vanilla(params, np.squeeze(h), np.squeeze(x.T), alpha)
        h_t.append(np.expand_dims(h,axis=1))

    h_t = np.squeeze(np.array(h_t))  
    return h_t

In [29]:
# for each initial condition in hs run the RNN with input x for N iterations
# return the last K hidden states
def get_attractor_points(hs, x, N, K):
  inputs = np.repeat(x, N).reshape(7, N).transpose()

  attracting_points = np.zeros((len(hs), K, 256))
  for i in range(len(hs)):
    attracting_points[i] = vanilla_run_with_h0(params, inputs, hs[i], alpha)[-K:]

  return attracting_points




In [71]:
# generate the input

fixation = 0
stimulus = 0 # 0 = stimulus off, 1 = stimulus 1, 2 = stimulus 2
task = 2    # ['fdgo', 'delaygo', 'fdanti', 'delayanti']
x = False
#x = [1,0,0,0,0,0,0]


if not x:
  x = np.zeros(7)
  x[0] = fixation
  if stimulus:
    x[stimulus] = 1.0
  x[task + 2] = 1.0



In [72]:
from tools_lnd import gen_trials_from_model_dir,make_FP_axs,take_names,same_stim_trial,\
gen_X_from_model_dir,get_T_inds,comp_eig_decomp

hiddens = {}
for rule in rule_trains:

  trial = gen_trials_from_model_dir(m, rule, mode = 'test', noise_on = False)
  _,hs = gen_X_from_model_dir(m,trial) # hs.shape = (n_rnn, 2, N), 2 because there are 2 stimuli
  hs = np.transpose(hs,(1,2,0))
  hiddens[rule] = hs.reshape(-1, n_rnn) # reshape to concatenate the hiddens for the 2 stimuli


Variables being optimized:
<tf.Variable 'rnn/leaky_rnn_cell/kernel:0' shape=(263, 256) dtype=float32_ref>
<tf.Variable 'rnn/leaky_rnn_cell/bias:0' shape=(256,) dtype=float32_ref>
<tf.Variable 'output/weights:0' shape=(256, 3) dtype=float32_ref>
<tf.Variable 'output/biases:0' shape=(3,) dtype=float32_ref>
INFO:tensorflow:Restoring parameters from /Users/slibkind/Documents/data/sophie-nets/binary_inputs/data/basic/LeakyRNN/softplus/randgauss/2_tasks/256_n_rnn/lr7.0l2_w7.0_h7.0_sig_rec0.05_sig_x0.1_w_rec_coeff0.9_delaygo_delayanti/0/model.ckpt
Model restored from file: /Users/slibkind/Documents/data/sophie-nets/binary_inputs/data/basic/LeakyRNN/softplus/randgauss/2_tasks/256_n_rnn/lr7.0l2_w7.0_h7.0_sig_rec0.05_sig_x0.1_w_rec_coeff0.9_delaygo_delayanti/0/model.ckpt
Variables being optimized:
<tf.Variable 'rnn/leaky_rnn_cell/kernel:0' shape=(263, 256) dtype=float32_ref>
<tf.Variable 'rnn/leaky_rnn_cell/bias:0' shape=(256,) dtype=float32_ref>
<tf.Variable 'output/weights:0' shape=(256, 3) dt

In [87]:
# get attractors
N = 500
K = 20

attracting_points = {}
for rule in rule_trains:
  attracting_points[rule] = get_attractor_points(hiddens[rule], x, N, K).reshape(-1,n_rnn)



In [100]:
with open('attracting_points.pickle', 'wb') as f:
    pickle.dump(np.vstack(attracting_points.values()).shape, f)